# 03 · Baseline Machine Learning

Goal:

1. Load the processed feature table from Google Drive.
2. Split the data chronologically.
3. Purge the final 5 trading days before each split boundary to prevent target overlap.
4. Establish a dummy baseline.
5. Train a Logistic Regression baseline.
6. Evaluate **validation data only**.

The 2026 test set is created but deliberately left untouched until final model comparison.

## Prediction task

Predict:

`Large_Move_5D = 1`

when the absolute return over the next five trading days is greater than 3%.

In [ ]:
from google.colab import drive
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

drive.mount("/content/drive")

PROJECT_ROOT = Path("/content/drive/MyDrive/ai-tech-market-risk")

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "metrics"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)

ML_DATA_FILE = PROCESSED_DATA_DIR / "ml_features.csv"

print("Project root:", PROJECT_ROOT)
print("Input:", ML_DATA_FILE)
print("scikit-learn:", sklearn.__version__)

## 1. Load the ML-ready dataset

In [ ]:
if not ML_DATA_FILE.exists():
    raise FileNotFoundError(
        f"Processed dataset not found: {ML_DATA_FILE}. "
        "Run Notebook 02 first."
    )

ml_data = pd.read_csv(
    ML_DATA_FILE,
    parse_dates=["Date"],
)

ml_data = (
    ml_data
    .sort_values(["Date", "Ticker"])
    .reset_index(drop=True)
)

print("Shape:", ml_data.shape)
print(
    "Date range:",
    ml_data["Date"].min().date(),
    "to",
    ml_data["Date"].max().date(),
)

ml_data.head()

## 2. Define features and target

`Future_Return_5D` is deliberately **not** included as a feature because it contains future information.

In [ ]:
FEATURE_COLUMNS = [
    "Return_1D",
    "Return_5D",
    "Return_10D",
    "Return_20D",
    "Volatility_5D",
    "Volatility_10D",
    "Volatility_20D",
    "Volume_Change_1D",
    "Relative_Volume_20D",
    "Price_vs_MA_5D",
    "Price_vs_MA_20D",
    "SPY_Return_1D",
    "QQQ_Return_1D",
    "SMH_Return_1D",
    "SPY_Return_5D",
    "QQQ_Return_5D",
    "SMH_Return_5D",
    "Excess_vs_QQQ_1D",
    "Excess_vs_SMH_1D",
]

CATEGORICAL_COLUMNS = ["Ticker"]

MODEL_COLUMNS = FEATURE_COLUMNS + CATEGORICAL_COLUMNS

TARGET_COLUMN = "Large_Move_5D"

required_columns = (
    ["Date", TARGET_COLUMN]
    + MODEL_COLUMNS
)

missing_columns = set(required_columns) - set(ml_data.columns)

if missing_columns:
    raise ValueError(
        f"Dataset is missing required columns: {sorted(missing_columns)}"
    )

if ml_data[MODEL_COLUMNS + [TARGET_COLUMN]].isna().any().any():
    raise ValueError("Missing values found in model features or target.")

if np.isinf(
    ml_data[FEATURE_COLUMNS].to_numpy()
).any():
    raise ValueError("Infinite numeric feature values detected.")

print("Numeric features:", len(FEATURE_COLUMNS))
print("Categorical features:", CATEGORICAL_COLUMNS)
print("Target:", TARGET_COLUMN)

## 3. Chronological split

We use calendar years for an easy-to-explain evaluation design:

* **Training:** 2018 to 2024
* **Validation:** 2025
* **Test:** 2026

Because the target looks five trading days forward, the final five trading dates in the training period are removed. Otherwise those training labels could use prices from the validation period.

The same purge is applied between validation and test.

In [ ]:
TARGET_HORIZON_DAYS = 5

VALIDATION_START = pd.Timestamp("2025-01-01")
TEST_START = pd.Timestamp("2026-01-01")

train_data = ml_data[
    ml_data["Date"] < VALIDATION_START
].copy()

validation_data = ml_data[
    (ml_data["Date"] >= VALIDATION_START)
    & (ml_data["Date"] < TEST_START)
].copy()

test_data = ml_data[
    ml_data["Date"] >= TEST_START
].copy()


def purge_last_trading_dates(data, n_dates):
    unique_dates = np.array(
        sorted(data["Date"].unique())
    )

    if len(unique_dates) <= n_dates:
        raise ValueError(
            "Not enough dates to apply the requested purge."
        )

    purged_dates = unique_dates[-n_dates:]

    cleaned = data[
        ~data["Date"].isin(purged_dates)
    ].copy()

    return cleaned, purged_dates


train_data, purged_train_dates = purge_last_trading_dates(
    train_data,
    TARGET_HORIZON_DAYS,
)

validation_data, purged_validation_dates = purge_last_trading_dates(
    validation_data,
    TARGET_HORIZON_DAYS,
)

print(
    "Purged training dates:",
    pd.to_datetime(purged_train_dates).date,
)

print(
    "Purged validation dates:",
    pd.to_datetime(purged_validation_dates).date,
)

## 4. Validate split boundaries

In [ ]:
split_summary = pd.DataFrame(
    {
        "split": [
            "train",
            "validation",
            "test",
        ],
        "rows": [
            len(train_data),
            len(validation_data),
            len(test_data),
        ],
        "start_date": [
            train_data["Date"].min(),
            validation_data["Date"].min(),
            test_data["Date"].min(),
        ],
        "end_date": [
            train_data["Date"].max(),
            validation_data["Date"].max(),
            test_data["Date"].max(),
        ],
        "positive_rate": [
            train_data[TARGET_COLUMN].mean(),
            validation_data[TARGET_COLUMN].mean(),
            test_data[TARGET_COLUMN].mean(),
        ],
    }
)

split_summary["positive_rate"] *= 100

split_summary

In [ ]:
if not (
    train_data["Date"].max()
    < validation_data["Date"].min()
    < test_data["Date"].min()
):
    raise ValueError("Chronological split ordering is invalid.")

train_dates = set(train_data["Date"])
validation_dates = set(validation_data["Date"])
test_dates = set(test_data["Date"])

if train_dates & validation_dates:
    raise ValueError("Train and validation dates overlap.")

if train_dates & test_dates:
    raise ValueError("Train and test dates overlap.")

if validation_dates & test_dates:
    raise ValueError("Validation and test dates overlap.")

print("Chronological split validation passed.")
print("2026 test set will NOT be evaluated in this notebook.")

## 5. Create train and validation matrices

We include `Ticker` as a categorical feature because AMD, NVDA, MSFT, and the other stocks have different natural large-move rates.

The ticker is one-hot encoded. Numeric features are standardized using statistics learned **only from the training set**.

In [ ]:
X_train = train_data[MODEL_COLUMNS].copy()
y_train = train_data[TARGET_COLUMN].astype(int).copy()

X_validation = validation_data[MODEL_COLUMNS].copy()
y_validation = validation_data[TARGET_COLUMN].astype(int).copy()

# Created now, but not used for evaluation yet.
X_test = test_data[MODEL_COLUMNS].copy()
y_test = test_data[TARGET_COLUMN].astype(int).copy()

print("X_train:", X_train.shape)
print("X_validation:", X_validation.shape)
print("X_test:", X_test.shape)

print("\nTraining positive rate:")
print(f"{y_train.mean() * 100:.2f}%")

print("\nValidation positive rate:")
print(f"{y_validation.mean() * 100:.2f}%")

## 6. Metric helper

For this project we will not rely only on accuracy.

* **Precision:** When the model predicts a large move, how often is it right?
* **Recall:** Of all actual large moves, how many did it detect?
* **F1:** Balance between precision and recall.
* **ROC AUC:** Ranking ability across classification thresholds.
* **PR AUC:** Particularly useful when the positive class becomes less common in later target definitions.

In [ ]:
def evaluate_classifier(
    name,
    model,
    X,
    y,
):
    predictions = model.predict(X)
    probabilities = model.predict_proba(X)[:, 1]

    metrics = {
        "model": name,
        "accuracy": accuracy_score(
            y,
            predictions,
        ),
        "precision": precision_score(
            y,
            predictions,
            zero_division=0,
        ),
        "recall": recall_score(
            y,
            predictions,
            zero_division=0,
        ),
        "f1": f1_score(
            y,
            predictions,
            zero_division=0,
        ),
        "roc_auc": roc_auc_score(
            y,
            probabilities,
        ),
        "pr_auc": average_precision_score(
            y,
            probabilities,
        ),
    }

    return metrics, predictions, probabilities

## 7. Dummy baseline

Before ML can be considered useful, it should beat a model that simply uses the training-set class distribution.

In [ ]:
dummy_model = DummyClassifier(
    strategy="prior"
)

dummy_model.fit(
    X_train,
    y_train,
)

dummy_metrics, dummy_predictions, dummy_probabilities = (
    evaluate_classifier(
        "DummyClassifier",
        dummy_model,
        X_validation,
        y_validation,
    )
)

pd.DataFrame([dummy_metrics])

## 8. Logistic Regression baseline

This is our first actual machine-learning model.

Why start here?

* Fast
* Interpretable
* Strong sanity-check baseline
* Much easier to debug than jumping directly into XGBoost or neural networks

In [ ]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(
    handle_unknown="ignore"
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            numeric_transformer,
            FEATURE_COLUMNS,
        ),
        (
            "ticker",
            categorical_transformer,
            CATEGORICAL_COLUMNS,
        ),
    ]
)

logistic_model = Pipeline(
    steps=[
        (
            "preprocess",
            preprocessor,
        ),
        (
            "model",
            LogisticRegression(
                max_iter=2000,
                solver="lbfgs",
                random_state=42,
            ),
        ),
    ]
)

logistic_model.fit(
    X_train,
    y_train,
)

print("Logistic Regression fitted.")

In [ ]:
logistic_metrics, logistic_predictions, logistic_probabilities = (
    evaluate_classifier(
        "LogisticRegression",
        logistic_model,
        X_validation,
        y_validation,
    )
)

validation_results = pd.DataFrame(
    [
        dummy_metrics,
        logistic_metrics,
    ]
).set_index("model")

validation_results.round(4)

## 9. Classification report

In [ ]:
print(
    classification_report(
        y_validation,
        logistic_predictions,
        target_names=[
            "Normal Move",
            "Large Move",
        ],
        digits=4,
    )
)

## 10. Confusion matrix

Rows are actual classes. Columns are predicted classes.

In [ ]:
cm = confusion_matrix(
    y_validation,
    logistic_predictions,
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=[
        "Normal Move",
        "Large Move",
    ],
)

disp.plot()
plt.title(
    "Logistic Regression - 2025 Validation"
)
plt.show()

## 11. Inspect Logistic Regression coefficients

Positive coefficients push the model toward predicting a large move. Negative coefficients push it toward predicting a normal move.

Coefficient magnitude should not be treated as causal importance.

In [ ]:
feature_names = (
    logistic_model
    .named_steps["preprocess"]
    .get_feature_names_out()
)

coefficients = (
    logistic_model
    .named_steps["model"]
    .coef_[0]
)

coefficient_table = pd.DataFrame(
    {
        "feature": feature_names,
        "coefficient": coefficients,
    }
)

print("Largest positive coefficients:")
display(
    coefficient_table
    .sort_values(
        "coefficient",
        ascending=False,
    )
    .head(10)
)

print("Largest negative coefficients:")
display(
    coefficient_table
    .sort_values(
        "coefficient",
        ascending=True,
    )
    .head(10)
)

## 12. Save validation metrics

We save only metrics here, not a "production" model. Model selection is not finished yet.

In [ ]:
METRICS_FILE = (
    REPORTS_DIR
    / "03_baseline_validation_metrics.csv"
)

validation_results.to_csv(
    METRICS_FILE
)

print("Saved:", METRICS_FILE)
print("Exists:", METRICS_FILE.exists())
print("Size:", METRICS_FILE.stat().st_size, "bytes")

In [ ]:
reloaded_metrics = pd.read_csv(
    METRICS_FILE,
    index_col=0,
)

print("Reloaded metrics:")
display(reloaded_metrics.round(4))

if reloaded_metrics.shape != validation_results.shape:
    raise ValueError(
        "Reloaded metrics file has an unexpected shape."
    )

print("Baseline ML pipeline completed successfully.")
print("2026 test set remains untouched.")

# What to send back

Send these outputs before we move to XGBoost:

1. `split_summary`
2. `validation_results`
3. Logistic Regression classification report
4. Confusion matrix values
5. Top positive and negative coefficients

Then we will decide whether Logistic Regression has learned anything beyond the dummy baseline before adding a stronger model.